# short 会话记忆 · Token 滑动窗口

```bash
pip install tiktoken openai python-dotenv
```

In [1]:
import sys
from dataclasses import dataclass
from typing import Dict, List, Literal

import tiktoken

try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, OSError):
    pass  # Jupyter OutStream 不支持 reconfigure

Role = Literal["system", "user", "assistant"]

### WindowedConversationMemory 算法逻辑图

**核心思想**：维护一个按 token 计量的滑动窗口；每次 `add` 后若总量超过 `max_tokens`，从列表**头部**循环 `pop(0)` 丢弃最旧消息，直到回到预算内。

```mermaid
flowchart TD
    START(["add(role, content)"])
    APPEND["messages.append(ChatMessage)"]
    SHRINK["进入 _shrink_to_budget()"]
    CHECK{"messages 非空<br/>且<br/>_count_tokens(messages) > max_tokens?"}
    POP["pop(0) 丢弃队首最旧消息"]
    LOG["打印 [budget] 丢弃日志"]
    DONE(["窗口收敛，结束"])
    OUT(["as_openai_messages() 输出给 LLM"])

    START --> APPEND --> SHRINK --> CHECK
    CHECK -->|是| POP --> LOG --> CHECK
    CHECK -->|否| DONE
    DONE -.->|下次推理前| OUT

    style START fill:#e3f2fd,stroke:#1976D2
    style CHECK fill:#fff8e1,stroke:#F57C00
    style POP fill:#ffebee,stroke:#c62828
    style DONE fill:#e8f5e9,stroke:#388E3C
```

**`_count_tokens` 子算法**

```mermaid
flowchart LR
    M1["遍历 messages"]
    M2["拼接为<br/>role: content\\n..."]
    M3["tiktoken.encode"]
    M4["返回 token 总数"]
    M1 --> M2 --> M3 --> M4
```



In [2]:
@dataclass
class ChatMessage:
    role: Role
    content: str


class WindowedConversationMemory:
    """Token 滑动窗口：超限则从头部 pop 最旧消息。"""

    def __init__(self, model: str = "gpt-4o", max_tokens: int = 3000):
        self.max_tokens = max_tokens
        _ = model  # 讲课时可说明不同模型应对应不同 encoding
        self.enc = tiktoken.get_encoding("cl100k_base")
        self.messages: List[ChatMessage] = []

    def add(self, role: Role, content: str) -> None:
        self.messages.append(ChatMessage(role=role, content=content))
        self._shrink_to_budget()

    def _count_tokens(self, messages: List[ChatMessage]) -> int:
        text = "\n".join(f"{m.role}: {m.content}" for m in messages)
        return len(self.enc.encode(text))

    def _shrink_to_budget(self) -> None:
        while self.messages and self._count_tokens(self.messages) > self.max_tokens:
            dropped = self.messages.pop(0)
            print(f"\n[budget] tokens > {self.max_tokens}，pop(0) 丢弃 [{dropped.role}] {dropped.content[:20]}...")

    def as_openai_messages(self) -> List[Dict[str, str]]:
        return [{"role": m.role, "content": m.content} for m in self.messages]

    def print_mem(self, title: str = "当前 memory") -> None:
        """打印 messages 列表及 token 计数，便于观察滑动窗口。"""
        print(f"--- {title} mem  ---")
        for msg in self.messages:
            print(f"[{msg.role}] {msg.content}")
        print(f"tokens ~= {self._count_tokens(self.messages)} / max {self.max_tokens}")

---
## 运行示例

需要配置 **`DEEPSEEK_API_KEY`** 或 **`OPENAI_API_KEY`**（可写在项目根目录 `.env`）。

下面用**真实 LLM** 多轮对话；设置 `max_tokens=80`，观察每次 `add()` 后 **按 token 从头部 pop** 的滑动效果：

In [3]:
import os
from pathlib import Path
from openai import OpenAI


def create_client() -> tuple[OpenAI, str]:
    """ODeepSeek"""
    deepseek_key = "sk-your-api-key-here"
    print("[LLM] DeepSeek deepseek-chat")
    return OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com"), "deepseek-chat"



client, model = create_client()

[LLM] DeepSeek deepseek-chat


In [9]:
mem = WindowedConversationMemory(max_tokens=200)
mem.add("system", "你是助手，回答尽量简洁。在100字以内")


def chat(mem: WindowedConversationMemory, client: OpenAI, model: str, user_text: str) -> str:
    """追加 user → OpenAI Chat Completions → 追加 assistant 回复。"""
    mem.add("user", user_text)
    response = client.chat.completions.create(
        model=model,
        messages=mem.as_openai_messages(),
        temperature=0,
    )
    content = (response.choices[0].message.content or "").strip()
    mem.add("assistant", content)
    return content


user_turns = [
    "我叫阿明。",
    "什么是llm",
    "上什么是transformer",
    "请解释mem",
]

for i, text in enumerate(user_turns, start=1):
    print(f"================= {i} turn start ======================")
    answer = chat(mem, client, model, text)
    print(f"[user] {text}")
    print(f"[assistant] {answer}\n")
    mem.print_mem(f"第 {i} 轮结束后")
    print(f"============= {i} turn end ==========================\n")

mem.print_mem("最终保留的消息")

================= 1 turn start ======================
[user] 我叫阿明。
[assistant] 阿明你好，我是助手。有什么需要帮忙的？请说。

--- 第 1 轮结束后 mem  ---
[system] 你是助手，回答尽量简洁。在100字以内
[user] 我叫阿明。
[assistant] 阿明你好，我是助手。有什么需要帮忙的？请说。
tokens ~= 64 / max 200
============= 1 turn end ==========================

================= 2 turn start ======================
[user] 什么是llm
[assistant] LLM 是“大型语言模型”的缩写，像 ChatGPT 这样的 AI，能理解和生成人类语言。

--- 第 2 轮结束后 mem  ---
[system] 你是助手，回答尽量简洁。在100字以内
[user] 我叫阿明。
[assistant] 阿明你好，我是助手。有什么需要帮忙的？请说。
[user] 什么是llm
[assistant] LLM 是“大型语言模型”的缩写，像 ChatGPT 这样的 AI，能理解和生成人类语言。
tokens ~= 111 / max 200
============= 2 turn end ==========================

================= 3 turn start ======================
[user] 上什么是transformer
[assistant] Transformer 是一种深度学习模型架构，核心是“自注意力机制”，能并行处理序列数据。它是 GPT 等 LLM 的基础。

--- 第 3 轮结束后 mem  ---
[system] 你是助手，回答尽量简洁。在100字以内
[user] 我叫阿明。
[assistant] 阿明你好，我是助手。有什么需要帮忙的？请说。
[user] 什么是llm
[assistant] LLM 是“大型语言模型”的缩写，像 ChatGPT 这样的 AI，能理解和生成人类语言。
[user] 上什么是transformer